# Upstream Bridge Notebook

This notebook connects the local companion to upstream materials:

- `notebooks/00_math_physics_preliminaries.ipynb`
- `references/upstream/implicit_layers_tutorial/chapter_1.ipynb`
- `references/upstream/implicit_layers_tutorial/chapter_2.ipynb`
- `references/upstream/implicit_layers_tutorial/chapter_3.ipynb`
- `references/upstream/implicit_layers_tutorial/chapter_4.ipynb`
- `references/upstream/implicit_layers_tutorial/chapter_5.ipynb`
- `references/upstream/locuslab_deq/repo/lib/solvers.py`
- `references/upstream/locuslab_deq/repo/lib/jacobian.py`
- `references/upstream/locuslab_deq/repo/MDEQ-Vision/`
- `references/upstream/recent_deq/torchdeq_repo/`
- `references/upstream/recent_deq/DEMPytorchMNIST_LT2025TMLR.ipynb`
- Numbered research references: https://jseluis.github.io/silva-networks/paper/references/

Use it as a guided index before diving into the upstream notebooks or code.


In [ ]:

from pathlib import Path
import json

ROOT = Path.cwd()
local = sorted((ROOT / "notebooks").glob("*.ipynb"))
upstream = sorted((ROOT / "references/upstream/implicit_layers_tutorial").glob("chapter_*.ipynb"))
print("Local companion notebooks:")
for p in local:
    if p.name != "00_upstream_bridge.ipynb":
        print(" -", p)
print("\nUpstream tutorial notebooks:")
for p in upstream:
    nb = json.loads(p.read_text())
    print(" -", p, "cells:", len(nb.get("cells", [])))


## Suggested Reading Order

1. Run local Chapter 0 first: notation, algebra, residuals, Jacobians, graph
   aggregation, mean-field global context, Hutchinson probes, and adjoints.
2. Run local Chapter 1 and upstream tutorial Chapter 1 side by side.
3. Run local Chapters 2-4 before upstream tutorial Chapter 2.
4. Run local Chapter 5 before upstream tutorial Chapter 4 and Locus Lab `lib/solvers.py`.
5. Run local Chapters 6-7 before reading Locus Lab `MDEQ-Vision/` and `lib/jacobian.py`.
6. Run local Chapters 8-15 for SILVA, small datasets, diagnostics, and the capstone.
7. Run local Chapters 16-24 for recent DEQ papers: PDE neural operators,
   homotopy, TorchDEQ/DeltaDEQ, certified robustness, score/diffusion models,
   DEQHNet, recent theory, distributional DEQs, and algorithmic/quantum reasoning.


In [ ]:

# Inspect upstream solver and Jacobian source snippets without importing them.
for rel in ["references/upstream/locuslab_deq/repo/lib/solvers.py",
            "references/upstream/locuslab_deq/repo/lib/jacobian.py"]:
    path = ROOT / rel
    print("\n==", rel, "==")
    if path.exists():
        lines = path.read_text(errors="replace").splitlines()
        for line in lines[:60]:
            print(line)
    else:
        print("missing; run git clone step or check references/upstream/INDEX.md")


## From 00 Upstream Bridge to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the tensor solved to equilibrium |
| Condition | the observed input or source tensor |
| Repeated computation | the state-preserving transition evaluated by the root solver |
| Required invariants | shape, device, dtype, finiteness, and differentiability |
| Replaceable components | initializer, source encoder, transition, readout, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**fixed-point residual and task error against a deterministic target**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **state width, batch size, and data volume**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '00_upstream_bridge.ipynb',
    "state": 'the tensor solved to equilibrium',
    "condition": 'the observed input or source tensor',
    "transition": 'the state-preserving transition evaluated by the root solver',
    "invariants": 'shape, device, dtype, finiteness, and differentiability',
    "compact_metric": 'fixed-point residual and task error against a deterministic target',
    "scale_axis": 'state width, batch size, and data volume',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record
